# 어텐션 (Attention)
입력 데이터 중에서 어떤 부분이 중요한지에 가중치(attention)를 주는 메커니즘이다.

사람이 문장을 읽을 때 중요한 단어에 더 집중하는 것처럼, 모델도 중요한 부분에 더 집중하도록 만든 기법이다.

## 왜 등장했을까?
기존 RNN/LSTM은 시퀀스가 길어지면 앞쪽 정보가 뒤쪽으로 잘 전달되지 않았다.

예) "나는 오늘 학교에서 친구들과 즐거운 시간을 보내고 집에 돌아와서 ___"<br>
    → 빈칸을 채우려면 "나는"이라는 주어 정보가 필요한데, RNN은 멀리 있는 정보를 잘 기억하지 못함

어텐션은 거리에 상관없이 모든 위치의 정보를 직접 참조할 수 있게 해준다.

## 핵심 개념: Query, Key, Value

| 용어 | 설명 | 비유 |
| --- | --- | --- |
| Query (Q) | 무엇을 찾고 있는가 | 검색어 |
| Key (K) | 각 데이터의 특징 | 검색 결과의 제목 |
| Value (V) | 실제 데이터 값 | 검색 결과의 내용 |

**과정:**
1. Query와 모든 Key의 유사도를 계산한다.
2. 유사도를 가중치로 사용한다.
3. 가중치를 Value에 곱해서 최종 출력을 만든다.

## Scaled Dot-Product Attention
$$Attention(Q, K, V) = softmax\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

- $QK^T$ : Query와 Key의 유사도 (내적)
- $\sqrt{d_k}$ : 차원이 클 때 값이 너무 커지는 것을 방지
- $softmax$ : 가중치를 0~1 사이 확률로 변환
- $V$ : 최종적으로 가중합을 계산

## PyTorch로 구현

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

def scaled_dot_product_attention(Q, K, V):
    d_k = Q.size(-1)

    # 1. Q와 K의 유사도 계산
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)

    # 2. softmax로 가중치화
    weights = F.softmax(scores, dim=-1)

    # 3. V에 가중치 적용
    output = torch.matmul(weights, V)

    return output, weights

# 예시
Q = torch.randn(1, 3, 4)  # (batch, seq_len, d_k)
K = torch.randn(1, 3, 4)
V = torch.randn(1, 3, 4)

output, weights = scaled_dot_product_attention(Q, K, V)
print("출력 shape:", output.shape)
print("가중치 shape:", weights.shape)

## Self-Attention
Query, Key, Value가 모두 같은 입력에서 나오는 어텐션

한 문장 내에서 단어들 간의 관계를 학습할 수 있다.

예) "나는 사과를 먹었다"에서 "먹었다"가 "사과"와 강한 관계를 가짐

## Multi-Head Attention
어텐션을 여러 번 병렬로 수행한 후 결과를 합치는 방식

각 헤드가 다른 관점으로 관계를 학습할 수 있어 표현력이 풍부해진다.

$$MultiHead(Q, K, V) = Concat(head_1, ..., head_h) W^O$$

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, Q, K, V):
        batch_size = Q.size(0)

        # 헤드 개수만큼 분할
        Q = self.W_q(Q).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(K).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(V).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        # 어텐션 수행
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        weights = F.softmax(scores, dim=-1)
        output = torch.matmul(weights, V)

        # 합치기
        output = output.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        output = self.W_o(output)

        return output

mha = MultiHeadAttention(d_model=64, num_heads=8)
x = torch.randn(2, 10, 64)
out = mha(x, x, x)  # Self-Attention
print(out.shape)